# 🧪 Sesión 07 · Especificación OpenAPI (Swagger) y ReDoc

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo puede un cliente saber qué endpoints existen sin leer tu código?

---

## 🖥️ Así se verá el resultado final

```json
GET /openapi.json  ->  HTTP 200

{
  "openapi": "3.1.0",
  "info": {
    "title": "Biblioteca API",
    "version": "1.0.0",
    "description": "API de libros del curso DAB-2201"
  },
  "paths": {
    "/api/libros": {
      "get":  {"summary": "Listar libros",  "responses": {"200": {...}}},
      "post": {"summary": "Crear un libro", "responses": {"201": {...}}}
    }
  },
  "components": {
    "schemas": {"Libro": {"type": "object", ...}}
  }
}

GET /docs   ->  HTTP 200 (Swagger UI)
GET /redoc  ->  HTTP 200 (ReDoc)
```

_La documentación no se escribe aparte: se genera del mismo código que atiende las peticiones._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Publicar una especificación OpenAPI viva.
- Documentar endpoints, modelos y códigos de respuesta.
- Verificar que Swagger UI y ReDoc consumen esa especificación.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] OpenAPI responde con metadatos → HTTP 200
- [ ] Swagger UI está disponible → HTTP 200
- [ ] ReDoc está disponible → HTTP 200
- [ ] El endpoint documentado responde → HTTP 200

## 🧩 El reto

Publica una API de libros con metadatos y modelos documentados. El contrato debe describir lo que el servidor realmente hace.

**Requisitos del contrato**

- La app se llama «Biblioteca API», versión 1.0.0, con una descripción.
- GET /openapi.json responde 200 y contiene las rutas /api/libros.
- El esquema `Libro` aparece en components.schemas porque un endpoint lo usa.
- GET /docs y GET /redoc responden 200.
- GET /api/libros responde 200 con el listado.

> **Pista conceptual:** En FastAPI, `title`, `version`, `description` y los modelos Pydantic alimentan OpenAPI automáticamente. Si el contrato y el código se contradicen, el contrato miente.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion07.md` con la captura del contrato.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [9]:

%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel, Field

# Configuración y metadatos de la API
app = FastAPI(
    title="Biblioteca API",
    version="1.0.0",
    description="API de libros del curso DAB-2201"
)

# Modelo de libro
class Libro(BaseModel):
    id: int
    titulo: str = Field(min_length=3, max_length=80)
    anio: int = Field(ge=1450, le=2100)
    precio: float = Field(gt=0)

# Modelo para crear libros
class LibroCreate(BaseModel):
    titulo: str = Field(min_length=3, max_length=80)
    anio: int = Field(ge=1450, le=2100)
    precio: float = Field(gt=0)

# Modelo de respuesta del listado
class LibrosResponse(BaseModel):
    libros: list[Libro]

# Lista vacía al iniciar la aplicación
libros = []

# Endpoint GET: consultar libros
@app.get(
    "/api/libros",
    summary="Listar libros",
    response_model=LibrosResponse,
    status_code=200
)
def listar_libros():
    return {"libros": libros}

# Endpoint POST: registrar un libro
@app.post(
    "/api/libros",
    summary="Crear un libro",
    response_model=Libro,
    status_code=201
)
def crear_libro(libro: LibroCreate):

    nuevo_libro = Libro(
        id=len(libros) + 1,
        titulo=libro.titulo,
        anio=libro.anio,
        precio=libro.precio
    )

    libros.append(nuevo_libro)

    return nuevo_libro


Overwriting app.py


In [3]:
from fastapi.testclient import TestClient
from app import app

cliente = TestClient(app)

print("OpenAPI:", cliente.get("/openapi.json").status_code)
print("Swagger:", cliente.get("/docs").status_code)
print("ReDoc:", cliente.get("/redoc").status_code)
print("Libros:", cliente.get("/api/libros").status_code)

OpenAPI: 200
Swagger: 200
ReDoc: 200
Libros: 200


In [4]:
contrato = cliente.get("/openapi.json").json()

print("Título:", contrato["info"]["title"])
print("Versión:", contrato["info"]["version"])
print("Descripción:", contrato["info"]["description"])

print("\nRutas disponibles:")
for ruta in contrato["paths"]:
    print(ruta)

print("\nModelo Libro:")
print(contrato["components"]["schemas"]["Libro"])

Título: Biblioteca API
Versión: 1.0.0
Descripción: API de libros del curso DAB-2201

Rutas disponibles:
/api/libros

Modelo Libro:
{'properties': {'id': {'type': 'integer', 'title': 'Id'}, 'titulo': {'type': 'string', 'maxLength': 80, 'minLength': 3, 'title': 'Titulo'}, 'anio': {'type': 'integer', 'maximum': 2100.0, 'minimum': 1450.0, 'title': 'Anio'}, 'precio': {'type': 'number', 'exclusiveMinimum': 0.0, 'title': 'Precio'}}, 'type': 'object', 'required': ['id', 'titulo', 'anio', 'precio'], 'title': 'Libro'}


In [5]:
# Prueba 1: OpenAPI
assert cliente.get("/openapi.json").status_code == 200
print("✅ OpenAPI responde correctamente")

# Prueba 2: Swagger
assert cliente.get("/docs").status_code == 200
print("✅ Swagger UI está disponible")

# Prueba 3: ReDoc
assert cliente.get("/redoc").status_code == 200
print("✅ ReDoc está disponible")

# Prueba 4: Listado de libros
respuesta = cliente.get("/api/libros")
assert respuesta.status_code == 200
print("✅ Listado de libros funciona")

print("\n🎉 Todas las pruebas fueron aprobadas")

✅ OpenAPI responde correctamente
✅ Swagger UI está disponible
✅ ReDoc está disponible
✅ Listado de libros funciona

🎉 Todas las pruebas fueron aprobadas


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Python 3.13, FastAPI y Pydantic v2, trabajando en Google Colab.|
| **Rol** | Desarrollador backend especializado en APIs RESTful y documentación OpenAPI. |
| **Especificación** | Crear una API llamada Biblioteca API, versión 1.0.0, con descripción, modelos documentados y endpoints GET y POST en /api/libros. El listado debe iniciar vacío y responder mediante un objeto JSON con la propiedad libros. |
| **Output** | Archivo app.py funcional, con documentación automática OpenAPI, Swagger UI y ReDoc, y respuestas HTTP correctas. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [10]:
from verificar import probar_http

casos = [{'descripcion': 'OpenAPI responde con metadatos',
  'metodo': 'GET',
  'ruta': '/openapi.json',
  'estado': 200,
  'json_contiene': {'openapi': '3.1.0',
                    'info': {'title': 'Biblioteca API', 'version': '1.0.0'},
                    'paths': {'/api/libros': {}},
                    'components': {'schemas': {'Libro': {}}}}},
 {'descripcion': 'Swagger UI está disponible',
  'metodo': 'GET',
  'ruta': '/docs',
  'estado': 200,
  'cuerpo_contiene': ['swagger']},
 {'descripcion': 'ReDoc está disponible', 'metodo': 'GET', 'ruta': '/redoc', 'estado': 200},
 {'descripcion': 'El endpoint documentado responde',
  'metodo': 'GET',
  'ruta': '/api/libros',
  'estado': 200,
  'json_contiene': {'libros': []}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. OpenAPI responde con metadatos
✅ 02. Swagger UI está disponible
✅ 03. ReDoc está disponible
✅ 04. El endpoint documentado responde
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion07.md` a `auditorias/auditoria_sesion07.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   R.- Lo que más me costó fue hacer que el listado de libros respondiera correctamente, porque al principio mostraba una lista directamente y el verificador esperaba un objeto JSON con la propiedad libros.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   R.- Decidí crear un modelo llamado LibrosResponse para organizar mejor la respuesta y dejar la lista vacía al iniciar el programa.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   R.- Detectó que estaba mostrando dos libros de ejemplo cuando debía iniciar sin libros. También detectó que la respuesta no tenía la estructura JSON esperada. Corregí esos detalles hasta conseguir las cuatro pruebas en verde.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
